# 📥 PMB UPI — PDF Download, Cleaning & RAG Chunking Pipeline

**Alur kerja:**
1. Install dependensi
2. Mount Google Drive
3. Download semua PDF dari pmb.upi.edu
4. Ekstrak teks (text PDF + OCR untuk scan)
5. Rename file berdasarkan konten
6. Cleaning teks
7. Chunking untuk RAG
8. Simpan hasil ke CSV / JSONL

## 🔧 CELL 1 — Install Semua Dependensi

In [ ]:
# Install semua library yang dibutuhkan
!pip install -q requests beautifulsoup4 tqdm pdfplumber
!apt-get install -y -q tesseract-ocr poppler-utils tesseract-ocr-ind
!pip install -q pytesseract pdf2image

print('✅ Semua dependensi berhasil diinstall')

## 💾 CELL 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Dataset_PMB_UPI'
os.makedirs(save_dir, exist_ok=True)
print(f'✅ Folder siap: {save_dir}')

## 🌐 CELL 3 — Scrape & Download PDF dari pmb.upi.edu

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm
import urllib3

# Matikan warning SSL (situs UPI kadang sertifikatnya bermasalah)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = 'https://pmb.upi.edu'
DOWNLOAD_PAGE = 'https://pmb.upi.edu/download'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def scrape_pdf_links(url):
    """Ambil semua link PDF dari halaman download."""
    try:
        response = requests.get(url, headers=HEADERS, verify=False, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f'❌ Gagal mengakses {url}: {e}')
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    pdf_links = []

    for a in soup.find_all('a', href=True):
        href = a['href']
        if '.pdf' in href.lower():
            full_url = urljoin(BASE_URL, href)
            if full_url not in pdf_links:
                pdf_links.append(full_url)

    return pdf_links


def download_pdfs(pdf_links, folder):
    """Download semua PDF ke folder tujuan."""
    downloaded, failed = [], []

    for link in tqdm(pdf_links, desc='Downloading PDFs'):
        try:
            filename = link.split('/')[-1].split('?')[0]  # hapus query string
            if not filename.lower().endswith('.pdf'):
                filename += '.pdf'

            filepath = os.path.join(folder, filename)

            # Skip jika sudah ada
            if os.path.exists(filepath):
                print(f'⏭️  Skip (sudah ada): {filename}')
                downloaded.append(filepath)
                continue

            res = requests.get(link, headers=HEADERS, verify=False, timeout=60)

            if res.status_code == 200 and len(res.content) > 1000:  # minimal 1KB
                with open(filepath, 'wb') as f:
                    f.write(res.content)
                downloaded.append(filepath)
            else:
                print(f'❌ Gagal download ({res.status_code}): {link}')
                failed.append(link)

        except Exception as e:
            print(f'❌ Error: {link} → {e}')
            failed.append(link)

    return downloaded, failed


# Jalankan scrape & download
print('🔍 Mencari link PDF...')
pdf_links = scrape_pdf_links(DOWNLOAD_PAGE)
print(f'📄 Total PDF ditemukan: {len(pdf_links)}')

if pdf_links:
    print('\n📥 Mulai download...')
    downloaded_files, failed_files = download_pdfs(pdf_links, save_dir)
    print(f'\n✅ Berhasil: {len(downloaded_files)} file')
    print(f'❌ Gagal   : {len(failed_files)} file')
else:
    print('⚠️  Tidak ada PDF ditemukan. Cek URL atau struktur halaman.')

## 📖 CELL 4 — Fungsi Ekstraksi Teks (Text PDF + OCR)

In [ ]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path

def is_scanned_pdf(filepath, sample_pages=3):
    """Cek apakah PDF adalah hasil scan (tidak ada layer teks)."""
    try:
        with pdfplumber.open(filepath) as pdf:
            pages_to_check = pdf.pages[:sample_pages]
            for page in pages_to_check:
                text = page.extract_text()
                if text and len(text.strip()) > 50:
                    return False  # Ada teks → bukan scan
        return True  # Tidak ada teks → kemungkinan scan
    except Exception:
        return True  # Jika error, anggap scan agar dicoba OCR


def ocr_pdf(filepath, lang='ind+eng'):
    """OCR untuk PDF hasil scan."""
    try:
        images = convert_from_path(filepath, dpi=200)
        text_all = ''
        for i, img in enumerate(images):
            text = pytesseract.image_to_string(img, lang=lang)
            text_all += f'[Halaman {i+1}]\n{text}\n'
        return text_all
    except Exception as e:
        return f'[OCR GAGAL: {e}]'


def extract_text_from_pdf(filepath):
    """Ekstrak teks dari PDF (text atau scan)."""
    scanned = is_scanned_pdf(filepath)

    if scanned:
        text = ocr_pdf(filepath)
        return text, 'scanned'
    else:
        try:
            with pdfplumber.open(filepath) as pdf:
                text = ''
                for i, page in enumerate(pdf.pages):
                    page_text = page.extract_text() or ''
                    text += f'[Halaman {i+1}]\n{page_text}\n'
            return text, 'text'
        except Exception as e:
            return f'[EKSTRAKSI GAGAL: {e}]', 'error'


print('✅ Fungsi ekstraksi teks siap')

## 🏷️ CELL 5 — Rename File Berdasarkan Konten

In [ ]:
import re

# Pemetaan kategori berdasarkan kata kunci (dapat ditambah sesuai kebutuhan)
KATEGORI_MAP = {
    'registrasi'  : ['registrasi', 'daftar ulang', 'pendaftaran'],
    'panduan'     : ['panduan', 'petunjuk', 'tata cara', 'prosedur'],
    'beasiswa'    : ['beasiswa', 'bantuan biaya', 'bidikmisi', 'kip'],
    'jadwal'      : ['jadwal', 'kalender', 'agenda'],
    'pengumuman'  : ['pengumuman', 'hasil seleksi', 'kelulusan'],
    'biaya'       : ['biaya', 'ukt', 'spp', 'pembayaran'],
    'formulir'    : ['formulir', 'form', 'blangko'],
    'syarat'      : ['syarat', 'persyaratan', 'ketentuan'],
}

def detect_category(text):
    text_lower = text.lower()
    for category, keywords in KATEGORI_MAP.items():
        if any(kw in text_lower for kw in keywords):
            return category
    return 'dokumen'


def detect_year(text):
    years = re.findall(r'20[1-3]\d', text)  # tahun 2010–2039
    return years[0] if years else 'unknown'


def generate_clean_filename(text, idx):
    category = detect_category(text)
    year = detect_year(text)
    return f'upi_{category}_{year}_{idx:03d}.pdf'


print('✅ Fungsi rename siap')

## 🧹 CELL 6 — Cleaning Teks

In [ ]:
def clean_text(text):
    """Bersihkan teks untuk keperluan RAG."""
    if not text:
        return ''

    # Hapus header halaman seperti [Halaman 1]
    text = re.sub(r'\[Halaman \d+\]', '', text)

    # Hapus karakter non-ASCII yang tidak diperlukan
    text = re.sub(r'[^\x00-\x7F\u00C0-\u024F\u0100-\u017F]+', ' ', text)

    # Normalisasi spasi & newline berlebih
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Hapus baris yang terlalu pendek (noise OCR)
    lines = text.split('\n')
    lines = [ln.strip() for ln in lines if len(ln.strip()) > 3]
    text = '\n'.join(lines)

    # Hapus nomor halaman soliter seperti "- 1 -" atau "halaman 1"
    text = re.sub(r'(?i)^[-–]?\s*halaman\s*\d+\s*[-–]?$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*[-–]\s*\d+\s*[-–]\s*$', '', text, flags=re.MULTILINE)

    return text.strip()


print('✅ Fungsi cleaning siap')

## ✂️ CELL 7 — Chunking untuk RAG

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    """
    Pecah teks menjadi chunk dengan overlap.
    - chunk_size : jumlah kata per chunk
    - overlap    : jumlah kata yang overlap antar chunk
    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunk_text_str = ' '.join(chunk_words)

        if len(chunk_text_str.strip()) > 20:  # abaikan chunk terlalu pendek
            chunks.append(chunk_text_str)

        if end >= len(words):
            break
        start = end - overlap  # geser dengan overlap

    return chunks


print('✅ Fungsi chunking siap')

## 🚀 CELL 8 — Jalankan Pipeline Utama (Rename + Clean + Chunk)

In [ ]:
import pandas as pd
import json

# Folder output untuk hasil chunking
output_dir = os.path.join(save_dir, 'output_rag')
os.makedirs(output_dir, exist_ok=True)

all_chunks = []      # untuk JSONL
rename_log = []      # untuk log rename
error_log  = []      # untuk log error

pdf_files = [f for f in os.listdir(save_dir) if f.endswith('.pdf')]
print(f'📂 Ditemukan {len(pdf_files)} file PDF di {save_dir}')

for idx, filename in enumerate(tqdm(pdf_files, desc='Memproses PDF')):
    old_path = os.path.join(save_dir, filename)

    try:
        # ── 1. Ekstrak teks ──────────────────────────────────────────
        raw_text, pdf_type = extract_text_from_pdf(old_path)

        if not raw_text or len(raw_text.strip()) < 20:
            print(f'⚠️  Teks kosong: {filename}')
            error_log.append({'file': filename, 'error': 'empty text'})
            continue

        # ── 2. Rename berdasarkan konten ──────────────────────────────
        new_filename = generate_clean_filename(raw_text, idx)
        new_path     = os.path.join(save_dir, new_filename)

        # Hindari overwrite jika nama sudah sama atau sudah ada
        if old_path != new_path:
            counter = 0
            base_new = new_filename[:-4]  # tanpa .pdf
            while os.path.exists(new_path):
                counter += 1
                new_path = os.path.join(save_dir, f'{base_new}_v{counter}.pdf')
                new_filename = os.path.basename(new_path)
            os.rename(old_path, new_path)

        rename_log.append({
            'old_name' : filename,
            'new_name' : new_filename,
            'type'     : pdf_type
        })

        # ── 3. Cleaning ───────────────────────────────────────────────
        clean = clean_text(raw_text)

        # ── 4. Chunking ───────────────────────────────────────────────
        chunks = chunk_text(clean, chunk_size=500, overlap=100)

        for chunk_idx, chunk in enumerate(chunks):
            all_chunks.append({
                'doc_id'     : new_filename,
                'chunk_id'   : f'{new_filename}__chunk_{chunk_idx:04d}',
                'chunk_index': chunk_idx,
                'total_chunks': len(chunks),
                'pdf_type'   : pdf_type,
                'text'       : chunk
            })

    except Exception as e:
        print(f'❌ Error memproses {filename}: {e}')
        error_log.append({'file': filename, 'error': str(e)})


# ── Simpan hasil ──────────────────────────────────────────────────────────

# 1. Log rename → CSV
df_rename = pd.DataFrame(rename_log)
rename_csv = os.path.join(output_dir, 'rename_log.csv')
df_rename.to_csv(rename_csv, index=False)

# 2. Semua chunk → CSV
df_chunks = pd.DataFrame(all_chunks)
chunks_csv = os.path.join(output_dir, 'chunks.csv')
df_chunks.to_csv(chunks_csv, index=False)

# 3. Semua chunk → JSONL (format umum untuk RAG)
jsonl_path = os.path.join(output_dir, 'chunks.jsonl')
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

# 4. Error log → CSV
if error_log:
    pd.DataFrame(error_log).to_csv(os.path.join(output_dir, 'error_log.csv'), index=False)

print('\n✅ Pipeline selesai!')
print(f'   📄 File diproses  : {len(rename_log)}')
print(f'   ✂️  Total chunks   : {len(all_chunks)}')
print(f'   ❌ Error          : {len(error_log)}')
print(f'\n📁 Output tersimpan di: {output_dir}')
print(f'   - rename_log.csv')
print(f'   - chunks.csv')
print(f'   - chunks.jsonl  ← gunakan ini untuk RAG')
if error_log:
    print(f'   - error_log.csv')

## 🔍 CELL 9 — Preview Hasil Chunking

In [ ]:
import pandas as pd

# Tampilkan preview chunks
df = pd.read_csv(os.path.join(output_dir, 'chunks.csv'))

print(f'📊 Total chunks : {len(df)}')
print(f'📄 Total dokumen: {df["doc_id"].nunique()}')
print(f'📝 Tipe PDF     : {df["pdf_type"].value_counts().to_dict()}')
print()
print('── Preview 3 chunk pertama ──')
for _, row in df.head(3).iterrows():
    print(f'\n[{row["chunk_id"]}]')
    print(row['text'][:300] + '...' if len(row['text']) > 300 else row['text'])
    print('─' * 60)

## 📊 CELL 10 — Statistik Dokumen per Kategori

In [ ]:
df_rename = pd.read_csv(os.path.join(output_dir, 'rename_log.csv'))

# Ekstrak kategori dari nama file baru
df_rename['category'] = df_rename['new_name'].str.extract(r'upi_(\w+)_')
df_rename['year']     = df_rename['new_name'].str.extract(r'_(20\d{2}|unknown)_')

print('📂 Distribusi kategori:')
print(df_rename['category'].value_counts().to_string())
print()
print('📅 Distribusi tahun:')
print(df_rename['year'].value_counts().to_string())
print()
print('📄 Tipe PDF:')
print(df_rename['type'].value_counts().to_string())